# GPT-2 124M - transferência de pesos e geração de texto

O notebook implementa os principais componentes do GPT em PyTorch, baixa o checkpoint público do GPT-2 124M, transfere os pesos e gera texto por amostragem top-k.


In [ ]:
import json
import urllib.error
import urllib.request
from pathlib import Path

import numpy as np
import tensorflow as tf
import tiktoken
import torch
import torch.nn as nn
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)


In [ ]:
def download_file(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return

    with urllib.request.urlopen(url) as response:
        total = int(response.headers.get("Content-Length", 0))
        with open(destination, "wb") as file, tqdm(
            total=total, unit="B", unit_scale=True, desc=destination.name
        ) as progress:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                file.write(chunk)
                progress.update(len(chunk))


def load_gpt2_params_from_tf_ckpt(ckpt_path: str, settings: dict) -> dict:
    params = {"blocks": [{} for _ in range(settings["n_layer"])]}

    for name, _ in tf.train.list_variables(ckpt_path):
        variable_array = np.squeeze(tf.train.load_variable(ckpt_path, name))
        parts = name.split("/")[1:]
        target = params

        if parts[0].startswith("h"):
            layer_number = int(parts[0][1:])
            target = params["blocks"][layer_number]

        for key in parts[1:-1]:
            target = target.setdefault(key, {})
        target[parts[-1]] = variable_array

    return params


def download_and_load_gpt2(model_size="124M", models_dir=Path("gpt2")):
    allowed_sizes = {"124M", "355M", "774M", "1558M"}
    if model_size not in allowed_sizes:
        raise ValueError(f"model_size deve estar em {sorted(allowed_sizes)}")

    model_dir = Path(models_dir) / model_size
    base_url = "https://openaipublic.blob.core.windows.net/gpt-2/models"
    filenames = [
        "checkpoint",
        "encoder.json",
        "hparams.json",
        "model.ckpt.data-00000-of-00001",
        "model.ckpt.index",
        "model.ckpt.meta",
        "vocab.bpe",
    ]

    for filename in filenames:
        url = f"{base_url}/{model_size}/{filename}"
        download_file(url, model_dir / filename)

    settings = json.loads((model_dir / "hparams.json").read_text())
    ckpt_path = tf.train.latest_checkpoint(str(model_dir))
    if ckpt_path is None:
        raise FileNotFoundError("Checkpoint do GPT-2 não foi encontrado.")

    params = load_gpt2_params_from_tf_ckpt(ckpt_path, settings)
    return settings, params


In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm + self.shift


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (
            1 + torch.tanh(
                torch.sqrt(torch.tensor(2.0 / torch.pi, device=x.device))
                * (x + 0.044715 * torch.pow(x, 3))
            )
        )


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        if d_out % num_heads != 0:
            raise ValueError("d_out deve ser divisível por num_heads")

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
        )

    def forward(self, x):
        batch_size, num_tokens, _ = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / (self.head_dim ** 0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = (attn_weights @ values).transpose(1, 2)
        context = context.contiguous().view(batch_size, num_tokens, self.d_out)
        return self.out_proj(context)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        return x + shortcut


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        _, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(tok_embeds + pos_embeds)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)


In [ ]:
def assign(left: torch.Tensor, right: np.ndarray) -> nn.Parameter:
    if left.shape != right.shape:
        raise ValueError(f"Shape incompatível: {left.shape} vs {right.shape}")
    return nn.Parameter(torch.tensor(right, dtype=left.dtype))


def load_weights_into_gpt(gpt: GPTModel, params: dict) -> None:
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])

    for block_index in range(len(params["blocks"])):
        block_params = params["blocks"][block_index]
        block = gpt.trf_blocks[block_index]

        q_w, k_w, v_w = np.split(block_params["attn"]["c_attn"]["w"], 3, axis=-1)
        q_b, k_b, v_b = np.split(block_params["attn"]["c_attn"]["b"], 3, axis=-1)

        block.att.W_query.weight = assign(block.att.W_query.weight, q_w.T)
        block.att.W_key.weight = assign(block.att.W_key.weight, k_w.T)
        block.att.W_value.weight = assign(block.att.W_value.weight, v_w.T)
        block.att.W_query.bias = assign(block.att.W_query.bias, q_b)
        block.att.W_key.bias = assign(block.att.W_key.bias, k_b)
        block.att.W_value.bias = assign(block.att.W_value.bias, v_b)

        block.att.out_proj.weight = assign(
            block.att.out_proj.weight,
            block_params["attn"]["c_proj"]["w"].T,
        )
        block.att.out_proj.bias = assign(
            block.att.out_proj.bias,
            block_params["attn"]["c_proj"]["b"],
        )

        block.ff.layers[0].weight = assign(
            block.ff.layers[0].weight,
            block_params["mlp"]["c_fc"]["w"].T,
        )
        block.ff.layers[0].bias = assign(
            block.ff.layers[0].bias,
            block_params["mlp"]["c_fc"]["b"],
        )
        block.ff.layers[2].weight = assign(
            block.ff.layers[2].weight,
            block_params["mlp"]["c_proj"]["w"].T,
        )
        block.ff.layers[2].bias = assign(
            block.ff.layers[2].bias,
            block_params["mlp"]["c_proj"]["b"],
        )

        block.norm1.scale = assign(block.norm1.scale, block_params["ln_1"]["g"])
        block.norm1.shift = assign(block.norm1.shift, block_params["ln_1"]["b"])
        block.norm2.scale = assign(block.norm2.scale, block_params["ln_2"]["g"])
        block.norm2.shift = assign(block.norm2.shift, block_params["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])


In [ ]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir=Path("gpt2"))

GPT_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.0,
    "qkv_bias": True,
}

model = GPTModel(GPT_CONFIG)
load_weights_into_gpt(model, params)
model.to(device)
model.eval()

tokenizer = tiktoken.get_encoding("gpt2")
print("Pesos carregados no modelo.")


In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=30, temperature=0.9, top_k=50):
    token_ids = tokenizer.encode(prompt)
    idx = torch.tensor(token_ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -GPT_CONFIG["context_length"]:]
        with torch.no_grad():
            logits = model(idx_cond)[:, -1, :]

        if temperature <= 0:
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k is not None:
                top_logits, top_indices = torch.topk(logits, min(top_k, logits.shape[-1]))
                probs = torch.softmax(top_logits, dim=-1)
                sampled_position = torch.multinomial(probs, num_samples=1)
                next_token = top_indices.gather(-1, sampled_position)
            else:
                probs = torch.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, next_token), dim=1)

    return tokenizer.decode(idx[0].tolist())


prompt = "Artificial intelligence can help"
print(generate_text(model, tokenizer, prompt, max_new_tokens=30, temperature=0.8, top_k=40))


## Observação

A geração é estocástica. Alterar `temperature` e `top_k` muda a diversidade do texto. Valores menores de temperatura tornam a escolha mais conservadora, enquanto valores maiores aumentam a variação.
